# CVPR T2: trần DC-AE f32 và SD-VAE f8 trên Kodak, CLIC2020 và DIV2K-val

**Cài đặt:** Accelerator = **GPU T4 x1**, Internet = **On**. Mất khoảng 1.5–2.5 giờ (tải ~2 GB; 552 ảnh × 2 AE; tính metric; FID/KID trên patch). Chạy bằng **Save Version → Save & Run All (Commit)** để notebook chạy nền; mỗi bước tự bỏ qua phần đã làm, nên hết giờ thì chạy lại là tiếp tục.

**Kết quả:** `ceiling_results.zip`, đặt vào `results/ceiling/`.

*Sinh bởi `notebooks/build.py` từ commit `4072286-dirty` của repo; đừng sửa tay notebook này.*

In [ ]:
!mkdir -p experiments/ceiling src/ratflow src/ratflow/eval

In [ ]:
%%writefile src/ratflow/__init__.py
"""Rate-as-Time flow-bridge image codec (CVPR 2027) and its CLIC 2027 submission."""


In [ ]:
%%writefile src/ratflow/eval/__init__.py


In [ ]:
%%writefile src/ratflow/eval/metrics.py
"""Shared evaluation code: every number in the paper and in the CLIC analyses goes through here,
so ceilings, baselines and our codec stay comparable.

Conventions
- Images are HxWx3 uint8 numpy arrays.
- PSNR per image from the RGB MSE. Dataset-level "CLIC PSNR" pools MSE weighted by pixel count.
- LPIPS (alex, inputs in [-1, 1]) and DISTS are computed on 512x512 tiles, weighted by tile area
  (tiles with a side < 64 are skipped), so 2K+ images fit in GPU memory.
- MS-SSIM on the full image (NaN if the short side is < 161, the minimum for 5 scales).
- FID/KID on 256x256 patches, HiFiC protocol: a grid plus a grid shifted by 128 in both directions.
- OCR CER: easyocr on the same crops (boxes detected on the original) for reference and reconstruction.

torch, lpips, piq, torchmetrics and easyocr are imported lazily so the numpy helpers work without them.
"""
from __future__ import annotations

import csv
import math
from pathlib import Path

import numpy as np
from PIL import Image

TILE = 512
FID_PATCH = 256
IMG_EXTS = (".png", ".jpg", ".jpeg")


# ---------------------------------------------------------------- io
def load(p) -> np.ndarray:
    return np.asarray(Image.open(p).convert("RGB"))


def save(a: np.ndarray, p):
    Path(p).parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(a).save(p)


def images(d: Path, exts=IMG_EXTS):
    """Sorted image files under d, skipping macOS zip debris (__MACOSX/, ._*)."""
    return sorted(p for p in Path(d).rglob("*") if p.suffix.lower() in exts
                  and not p.name.startswith("._") and "__MACOSX" not in p.parts)


def write_csv(rows, path):
    keys = list(dict.fromkeys(k for r in rows for k in r))
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, keys)
        w.writeheader()
        w.writerows(rows)


def read_csv(path):
    with open(path) as f:
        return list(csv.DictReader(f))


# ---------------------------------------------------------------- torch helpers
def device():
    import torch
    return "cuda" if torch.cuda.is_available() else "cpu"


def to_tensor(a: np.ndarray, dev):
    """HxWx3 uint8 -> 1x3xHxW float in [0, 1]."""
    import torch
    return torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1)[None].float().div(255).to(dev)


# ---------------------------------------------------------------- fidelity
def mse(a: np.ndarray, b: np.ndarray) -> float:
    return float(((a.astype(np.float64) - b) ** 2).mean())


def psnr_from_mse(m: float) -> float:
    return 10 * math.log10(255 ** 2 / max(m, 1e-10))


def psnr(a, b) -> float:
    return psnr_from_mse(mse(a, b))


def pooled_psnr(mses, pixels) -> float:
    """CLIC-style dataset PSNR: MSE averaged with pixel-count weights."""
    return psnr_from_mse(sum(m * n for m, n in zip(mses, pixels)) / sum(pixels))


def msssim(a, b, dev) -> float:
    import piq
    import torch
    if min(a.shape[:2]) < 161:
        return float("nan")
    with torch.no_grad():
        return float(piq.multi_scale_ssim(to_tensor(a, dev), to_tensor(b, dev), data_range=1.0))


# ---------------------------------------------------------------- perceptual
def tiled(fn, a, b, dev, tile=TILE) -> float:
    """Area-weighted mean of fn(tile_a, tile_b) over tile x tile crops; fn takes [0,1] tensors."""
    import torch
    H, W = a.shape[:2]
    tot = wsum = 0.0
    for y in range(0, H, tile):
        for x in range(0, W, tile):
            pa, pb = a[y:y + tile, x:x + tile], b[y:y + tile, x:x + tile]
            if min(pa.shape[:2]) < 64:
                continue
            n = pa.shape[0] * pa.shape[1]
            with torch.no_grad():
                tot += float(fn(to_tensor(pa, dev), to_tensor(pb, dev))) * n
            wsum += n
    return tot / wsum


class Perceptual:
    """LPIPS-alex and DISTS with the tiling convention above."""

    def __init__(self, dev=None):
        import lpips
        import piq
        self.dev = dev or device()
        self._lpips = lpips.LPIPS(net="alex", verbose=False).to(self.dev)
        self._dists = piq.DISTS().to(self.dev)

    def lpips(self, a, b) -> float:
        return tiled(lambda x, y: self._lpips(x * 2 - 1, y * 2 - 1).mean(), a, b, self.dev)

    def dists(self, a, b) -> float:
        return tiled(lambda x, y: self._dists(x, y), a, b, self.dev)

    def all(self, a, b) -> dict:
        m = mse(a, b)
        return dict(mse=m, psnr=psnr_from_mse(m), msssim=msssim(a, b, self.dev),
                    lpips=self.lpips(a, b), dists=self.dists(a, b))


# ---------------------------------------------------------------- FID / KID
def patches(a: np.ndarray, size=FID_PATCH):
    """HiFiC protocol: size x size grid, plus the grid shifted by size // 2."""
    H, W = a.shape[:2]
    for off in (0, size // 2):
        for y in range(off, H - size + 1, size):
            for x in range(off, W - size + 1, size):
                yield a[y:y + size, x:x + size]


def fid_kid(pairs, dev=None, batch=64, kid_subset=1000) -> dict:
    """pairs: iterable of (original, reconstruction) uint8 arrays. Returns fid, kid, kid_std, n_patches."""
    import torch
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.kid import KernelInceptionDistance
    dev = dev or device()
    f = FrechetInceptionDistance(feature=2048, normalize=False).to(dev)
    k = KernelInceptionDistance(subset_size=kid_subset, normalize=False).to(dev)

    def push(ps, real):
        t = torch.from_numpy(np.stack(ps)).permute(0, 3, 1, 2).to(dev)  # uint8 NCHW
        f.update(t, real=real)
        k.update(t, real=real)

    n = 0
    for x, y in pairs:
        pr, pf = list(patches(x)), list(patches(y))
        for i in range(0, len(pr), batch):
            push(pr[i:i + batch], True)
            push(pf[i:i + batch], False)
        n += len(pr)
    km, ks = k.compute()
    return dict(fid=float(f.compute()), kid=float(km), kid_std=float(ks), n_patches=n)


# ---------------------------------------------------------------- text legibility (OCR)
def edit_distance(a: str, b: str) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def ocr_reader():
    import easyocr
    import torch
    return easyocr.Reader(["en"], gpu=torch.cuda.is_available(), verbose=False)


def box_xyxy(bbox, H, W, pad=4):
    """easyocr quadrilateral -> padded, clipped axis-aligned box (x0, y0, x1, y1)."""
    xs = [p[0] for p in bbox]
    ys = [p[1] for p in bbox]
    return (max(0, int(min(xs)) - pad), max(0, int(min(ys)) - pad),
            min(W, int(max(xs)) + pad), min(H, int(max(ys)) + pad))


def detect_text(reader, img, min_conf=0.4, min_len=2):
    """[(box_xyxy, text)] for confident detections on the original image."""
    H, W = img.shape[:2]
    return [(box_xyxy(b, H, W), t) for b, t, c in reader.readtext(img) if c > min_conf and len(t.strip()) >= min_len]


def read_boxes(reader, img, boxes):
    return ["".join(t for _, t, _ in reader.readtext(img[y0:y1, x0:x1])) for x0, y0, x1, y1 in boxes]


def cer(refs, hyps) -> float:
    """Character error rate pooled over boxes; NaN if the reference has no characters."""
    n = sum(len(r) for r in refs)
    return sum(edit_distance(r, h) for r, h in zip(refs, hyps)) / n if n else float("nan")


def region_psnr(a, b, boxes) -> float:
    mask = np.zeros(a.shape[:2], bool)
    for x0, y0, x1, y1 in boxes:
        mask[y0:y1, x0:x1] = True
    return psnr_from_mse(float(((a[mask].astype(np.float64) - b[mask]) ** 2).mean())) if mask.any() else float("nan")


In [ ]:
%%writefile experiments/ceiling/ceiling.py
"""Autoencoder ceilings for the RD plots: DC-AE f32c32 (our backbone) vs SD-VAE f8 (StableCodec's).

An autoencode round trip without quantisation is the best any latent codec on that AE can do,
so it is drawn as a horizontal line on every RD plot.

Datasets: kodak (24), clic2020 (professional + mobile test = 428), div2k (DIV2K valid HR, 100).
Metrics per image: PSNR, MS-SSIM, LPIPS (alex), DISTS (both tiled 512).
FID / KID on 256x256 patches, HiFiC protocol (grid + grid shifted by 128), for clic2020 and div2k.

Steps: download, recon, metrics, fid, summary   (or: all). Each step skips work already done.
python ceiling.py all
"""
from __future__ import annotations

import argparse
import json
import subprocess
import sys
import zipfile
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "src"))  # repo checkout without `pip install -e .`
from ratflow.eval.metrics import (  # noqa: E402
    Perceptual, device, fid_kid, images, load, read_csv, save, to_tensor, write_csv,
)

DATASETS = {
    "kodak": [f"http://r0k.us/graphics/kodak/kodak/kodim{i:02d}.png" for i in range(1, 25)],
    "clic2020": ["https://downloads.compression.cc/clic2020_professional_test.zip",
                 "https://downloads.compression.cc/clic2020_mobile_test.zip"],
    "div2k": ["https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip"],
}
EXPECTED = {"kodak": 24, "clic2020": 428, "div2k": 100}
FID_SETS = ["clic2020", "div2k"]
CFGS = {  # name -> (repo, subfolder, class, spatial multiple)
    "dcae": [("Efficient-Large-Model/Sana_1600M_1024px_diffusers", "vae", "AutoencoderDC", 32)],
    "sdvae": [("stabilityai/sd-turbo", "vae", "AutoencoderKL", 8),
              ("stabilityai/sd-vae-ft-ema", None, "AutoencoderKL", 8)],
}


# ---------------------------------------------------------------- download
def download(args):
    for ds, urls in DATASETS.items():
        d = args.data / ds
        if d.exists() and len(images(d)) >= EXPECTED[ds]:
            print(ds, "ok", len(images(d)))
            continue
        d.mkdir(parents=True, exist_ok=True)
        for u in urls:
            dst = d / u.split("/")[-1]
            if not dst.exists():
                subprocess.run(["wget", "-q", "-O", str(dst), u], check=True)
            if dst.suffix == ".zip":
                with zipfile.ZipFile(dst) as z:
                    z.extractall(d)
                dst.unlink()
        n = len(images(d))
        print(ds, n, "images" + ("" if n == EXPECTED[ds] else f"  WARNING: expected {EXPECTED[ds]}"))


# ---------------------------------------------------------------- recon
def load_vae(cfg):
    import diffusers
    import torch
    for repo, sub, cls, mult in CFGS[cfg]:
        try:
            vae = getattr(diffusers, cls).from_pretrained(repo, subfolder=sub, torch_dtype=torch.float32)
            print(cfg, "->", repo)
            return vae, mult, repo
        except Exception as e:  # gated / missing -> fallback
            print("skip", repo, type(e).__name__)
    raise RuntimeError(f"no weights for {cfg}")


def recon(args):
    import torch
    d = device()
    info = {}
    for cfg in CFGS:
        vae, mult, repo = load_vae(cfg)
        info[cfg] = repo
        vae = vae.to(d).eval()
        vae.enable_tiling()
        for ds in DATASETS:
            for p in images(args.data / ds):
                dst = args.out / "recon" / ds / cfg / f"{p.stem}.png"
                if dst.exists():
                    continue
                x = load(p)
                H, W = x.shape[:2]
                t = to_tensor(x, d).mul(2).sub(1)
                t = torch.nn.functional.pad(t, (0, (-W) % mult, 0, (-H) % mult), mode="reflect")
                with torch.no_grad():
                    z = vae.encode(t)
                    z = z.latent if cfg == "dcae" else z.latent_dist.mode()
                    y = vae.decode(z).sample
                y = y[..., :H, :W].clamp(-1, 1).add(1).mul(127.5).round().byte()[0].permute(1, 2, 0).cpu().numpy()
                save(y, dst)
            print(cfg, ds, "done")
        del vae
        torch.cuda.empty_cache()
    args.out.mkdir(parents=True, exist_ok=True)
    json.dump(info, open(args.out / "weights_used.json", "w"), indent=1)


# ---------------------------------------------------------------- per-image metrics
def metrics(args):
    perc = Perceptual(device())
    for ds in DATASETS:
        path = args.out / f"metrics_{ds}.csv"
        if path.exists():
            continue
        rows = []
        for p in images(args.data / ds):
            x = load(p)
            for cfg in CFGS:
                q = args.out / "recon" / ds / cfg / f"{p.stem}.png"
                if not q.exists():
                    continue
                rows.append(dict(dataset=ds, name=p.stem, cfg=cfg, H=x.shape[0], W=x.shape[1],
                                 **perc.all(x, load(q))))
        write_csv(rows, path)
        print(ds, len(rows), "rows")


# ---------------------------------------------------------------- FID / KID (HiFiC patch protocol)
def fid(args):
    path = args.out / "fid.json"
    res = json.load(open(path)) if path.exists() else {}
    for ds in FID_SETS:
        for cfg in CFGS:
            key = f"{ds}/{cfg}"
            if key in res:
                continue
            pairs = ((load(p), load(q)) for p in images(args.data / ds)
                     if (q := args.out / "recon" / ds / cfg / f"{p.stem}.png").exists())
            res[key] = fid_kid(pairs, device())
            print(key, res[key])
            json.dump(res, open(path, "w"), indent=1)


# ---------------------------------------------------------------- summary
def summary(args):
    fidr = json.load(open(args.out / "fid.json")) if (args.out / "fid.json").exists() else {}
    rows_out = []
    L = ["# Trần autoencoder (encode rồi decode, không lượng tử)", "",
         "PSNR/MS-SSIM/LPIPS/DISTS lấy trung bình theo ảnh. FID/KID tính trên patch 256 (lưới cộng lưới lệch 128, theo giao thức HiFiC).", "",
         "| dataset | AE | PSNR | MS-SSIM | LPIPS↓ | DISTS↓ | FID↓ | KID×10³↓ |", "|---|---|---|---|---|---|---|---|"]
    for ds in DATASETS:
        p = args.out / f"metrics_{ds}.csv"
        if not p.exists():
            continue
        rs = read_csv(p)
        for cfg in CFGS:
            r = [x for x in rs if x["cfg"] == cfg]
            if not r:
                continue
            m = {k: float(np.nanmean([float(x[k]) for x in r])) for k in ("psnr", "msssim", "lpips", "dists")}
            fr = fidr.get(f"{ds}/{cfg}", {})
            row = dict(dataset=ds, ae=cfg, n=len(r), **m, fid=fr.get("fid", float("nan")),
                       kid=fr.get("kid", float("nan")))
            rows_out.append(row)
            L.append(f"| {ds} | {cfg} | {m['psnr']:.2f} | {m['msssim']:.4f} | {m['lpips']:.4f} | {m['dists']:.4f} "
                     f"| {row['fid']:.2f} | {row['kid'] * 1e3:.2f} |")
    write_csv(rows_out, args.out / "ceiling_summary.csv")  # -> horizontal lines on the RD plots
    L += ["", "`ceiling_summary.csv` dùng để vẽ đường trần nằm ngang trên các RD plot.",
          "Khi đánh giá codec của mình và các baseline, **dùng lại đúng `ratflow.eval.metrics`** (`Perceptual.all`, `fid_kid`), để số liệu so sánh được với nhau."]
    (args.out / "summary.md").write_text("\n".join(L), encoding="utf-8")
    print("\n".join(L))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("step", choices=["download", "recon", "metrics", "fid", "summary", "all"])
    ap.add_argument("--data", type=Path, default=Path("data"))
    ap.add_argument("--out", type=Path, default=Path("ceiling_out"))
    args = ap.parse_args()
    for s in (["download", "recon", "metrics", "fid", "summary"] if args.step == "all" else [args.step]):
        print(f"===== {s}")
        globals()[s](args)


if __name__ == "__main__":
    main()


In [ ]:
!pip install -q -U diffusers accelerate lpips piq torchmetrics torch-fidelity
!nvidia-smi --query-gpu=name,memory.total --format=csv; df -h /kaggle/working | tail -1

In [ ]:
C = 'experiments/ceiling/ceiling.py'
!python {C} download

In [ ]:
!python {C} recon

In [ ]:
!python {C} metrics

In [ ]:
!python {C} fid

In [ ]:
!python {C} summary

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('ceiling_out/summary.md', encoding='utf-8').read()))

In [ ]:
!cd ceiling_out && zip -q ../ceiling_results.zip *.csv *.json *.md && ls -la ../ceiling_results.zip